# Tokenization Method Comparison

This notebook compares how several widely used tokenizer families split the same inputs into tokens and token IDs. The goal is to make the differences concrete: WordPiece tokenizers, byte-level BPE tokenizers, SentencePiece tokenizers, and LLaMA-style tokenizers can produce very different sequences even when the original text is identical.

In [16]:
# Load the Hugging Face tokenizer interface and notebook display helpers.
from transformers import AutoTokenizer
from pathlib import Path
from IPython.display import Markdown, display
import json

## Environment Setup

The tokenizers are downloaded through Hugging Face and cached locally under `./models/`. A local configuration file provides the access token needed for gated models such as LLaMA.

In [17]:
# Keep downloaded tokenizer files local to this note so repeated runs are fast.
cache_dir = Path("./models/")

read_json = json.loads(open("./config.json").read())
hf_token = read_json["hg_access_token"]

## Example Inputs and Tokenizers

The examples are intentionally small but cover three important cases: a normal sentence, a morphologically rich word, and a sentence containing an emoji. The selected models represent common tokenizer designs used across encoder-only, decoder-only, and sequence-to-sequence language models.

In [18]:
# Compare plain text, subword splitting, and Unicode/emoji handling.
texts = [
    "Tokenization is not trivial.",
    "unbelievable",
    "I love machine learning 😊",
]

models = {
    "BERT WordPiece": "bert-base-uncased",
    "GPT-2 Byte-level BPE": "gpt2",
    "RoBERTa Byte-level BPE": "roberta-base",
    "T5 SentencePiece": "t5-small",
    "LLaMA-style tokenizer": "meta-llama/Llama-2-7b-hf",
    "Qwen-small": "Qwen/Qwen3-0.6B"
}


## Token and ID Comparison

For each tokenizer, the notebook records both the visible token strings and their numeric IDs. The token strings are useful for understanding segmentation behavior, while the IDs show what the model actually receives as discrete input before embedding lookup.

In [12]:
# Collect the tokenization result for every model-input pair.
comparison_tables = {text: [] for text in texts}

for name, model_id in models.items():
    tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir, token=hf_token)

    for text in texts:
        ids = tokenizer.encode(text, add_special_tokens=True)
        tokens = tokenizer.convert_ids_to_tokens(ids)

        comparison_tables[text].append({
            "Model": name,
            "Model ID": model_id,
            "TOKENS": tokens,
            "IDs": ids,
        })

# Escape Markdown-sensitive characters so tokens render correctly in tables.
def format_tokens(tokens):
    return "`" + " ".join(t.replace("|", "\\|").replace("`", "\\`") for t in tokens) + "`"

def format_ids(ids):
    return "`" + " ".join(map(str, ids)) + "`"

# Render one compact Markdown table for each example input.
for text, rows in comparison_tables.items():
    table = [
        f"### Text: `{text}`",
        "",
        "| Model | TOKENS | IDs |",
        "|---|---|---|",
    ]

    for row in rows:
        table.append(
            f"| {row['Model']} | {format_tokens(row['TOKENS'])} | {format_ids(row['IDs'])} |"
        )

    display(Markdown("\n".join(table)))

### Text: `Tokenization is not trivial.`

| Model | TOKENS | IDs |
|---|---|---|
| BERT WordPiece | `[CLS] token ##ization is not trivial . [SEP]` | `101 19204 3989 2003 2025 20610 1012 102` |
| GPT-2 Byte-level BPE | `Token ization Ġis Ġnot Ġtrivial .` | `30642 1634 318 407 20861 13` |
| RoBERTa Byte-level BPE | `<s> Token ization Ġis Ġnot Ġtrivial . </s>` | `0 45643 1938 16 45 30063 4 2` |
| T5 SentencePiece | `▁To ken ization ▁is ▁not ▁trivia l . </s>` | `304 2217 1707 19 59 22377 40 5 1` |
| LLaMA-style tokenizer | `<s> ▁Token ization ▁is ▁not ▁trivial .` | `1 25159 2133 338 451 12604 29889` |
| Qwen-small | `Token ization Ġis Ġnot Ġtrivial .` | `3323 2022 374 537 35647 13` |

### Text: `unbelievable`

| Model | TOKENS | IDs |
|---|---|---|
| BERT WordPiece | `[CLS] unbelievable [SEP]` | `101 23653 102` |
| GPT-2 Byte-level BPE | `un bel iev able` | `403 6667 11203 540` |
| RoBERTa Byte-level BPE | `<s> un bel iev able </s>` | `0 879 8494 18421 868 2` |
| T5 SentencePiece | `▁unbelievable </s>` | `25525 1` |
| LLaMA-style tokenizer | `<s> ▁un bel iev able` | `1 443 6596 10384 519` |
| Qwen-small | `un belie vable` | `359 31798 23760` |

### Text: `I love machine learning 😊`

| Model | TOKENS | IDs |
|---|---|---|
| BERT WordPiece | `[CLS] i love machine learning [UNK] [SEP]` | `101 1045 2293 3698 4083 100 102` |
| GPT-2 Byte-level BPE | `I Ġlove Ġmachine Ġlearning ĠðŁĺ Ĭ` | `40 1842 4572 4673 30325 232` |
| RoBERTa Byte-level BPE | `<s> I Ġlove Ġmachine Ġlearning ĠðŁĺ Ĭ </s>` | `0 100 657 3563 2239 17841 27969 2` |
| T5 SentencePiece | `▁I ▁love ▁machine ▁learning ▁ <unk> </s>` | `27 333 1437 1036 3 2 1` |
| LLaMA-style tokenizer | `<s> ▁I ▁love ▁machine ▁learning ▁ <0xF0> <0x9F> <0x98> <0x8A>` | `1 306 5360 4933 6509 29871 243 162 155 141` |
| Qwen-small | `I Ġlove Ġmachine Ġlearning ĠðŁĺ Ĭ` | `40 2948 5662 6832 26525 232` |

## Discussion of the Emoji Example

The final table shows why tokenization is not only about splitting words. For the sentence `I love machine learning 😊`, the ordinary words are handled in a mostly predictable way, but the emoji exposes a major difference between tokenizer designs. BERT and T5 fall back to unknown-token behavior (`[UNK]` or `<unk>`), which means the specific emoji is not preserved as a distinct symbol. In contrast, GPT-2, RoBERTa, Qwen, and LLaMA-style tokenizers preserve the input by breaking the emoji into byte-level or byte-like pieces. This keeps the text representable even when the tokenizer has never seen that exact character as a standalone token.

This example also illustrates the trade-off between vocabulary coverage and sequence length. Unknown-token approaches produce shorter sequences, but they discard information: many unseen characters collapse into the same generic token. Byte-level methods usually create more tokens for unusual Unicode characters, yet they retain enough detail for the model to distinguish one symbol from another. That is one reason modern LLM tokenizers often prefer byte-level or byte-fallback behavior, especially for multilingual text, emojis, code, and other inputs that contain characters outside a narrow training vocabulary.